# 03 – Open-Prompt-Injection preprocessing
**Dataset:** Open-Prompt-Injection benchmark platform (Liu et al., USENIX Security 2024, ref. [2])
**Source:** https://github.com/liu00222/Open-Prompt-Injection (MIT licence)
**Project:** Cyber Nexus

The repository contains **no task data of its own**: it ships (a) the instruction prompts for the 7 target/injected tasks and (b) loaders that download the task datasets at run time. This notebook processes what can be obtained here:

* **Part A** – the 46 task-instruction prompt files in the repo.
* **Part B** – **HSOL** (hate-speech / offensive-language tweets, Davidson et al.), one of the benchmark's task datasets, hosted on GitHub.

The other task datasets (SST-2, MRPC, RTE, SMS Spam, Gigaword, JFLEG) are downloaded from UCI / GLUE / Hugging Face hosts that this environment cannot reach; see the final cell for their sources.

**Content note:** HSOL consists of real tweets with offensive and hateful language. It is processed as text only; nothing is generated from it.

In [1]:
import json, ast, re, unicodedata, hashlib, subprocess, html
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [1]:


SEED = 42
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
md5 = lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()

ZW = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def clean_text(t, collapse_spaces=False):
    t = unicodedata.normalize("NFC", str(t))
    t = ZW.sub("", t); t = CTRL.sub("", t)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = "\n".join(line.rstrip() for line in t.split("\n"))
    t = re.sub(r"\n{3,}", "\n\n", t)
    if collapse_spaces: t = re.sub(r"[ \t]{2,}", " ", t)
    return t.strip()

def deva_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum("\u0900" <= c <= "\u097f" for c in letters) / max(len(letters), 1)

def lang_tag(t):
    r = deva_ratio(t)
    return "hi" if r > 0.5 else ("mixed" if r > 0.02 else "en")   # Latin-script Hinglish needs a separate detector

def group_split(df, group_col, strat_col=None, seed=SEED):
    """70/15/15 split where all rows of one group stay together (avoids leakage)."""
    cols = [group_col] + ([strat_col] if strat_col else [])
    g = df.drop_duplicates(group_col)[cols]
    s = g[strat_col] if strat_col else None
    g_tr, g_tmp = train_test_split(g, test_size=0.30, stratify=s, random_state=seed)
    s2 = g_tmp[strat_col] if strat_col else None
    g_va, g_te = train_test_split(g_tmp, test_size=0.50, stratify=s2, random_state=seed)
    parts = [df[df[group_col].isin(x[group_col])].copy().assign(split=n) for x, n in [(g_tr, "train"), (g_va, "val"), (g_te, "test")]]
    for a in range(3):
        for b in range(a + 1, 3):
            assert not (set(parts[a][group_col]) & set(parts[b][group_col])), "group leakage!"
    assert sum(len(p) for p in parts) == len(df)
    return parts

def add_buckets(parts, len_col, by=None):
    """short/medium/long using tertiles computed on the TRAIN part only."""
    tr = parts[0]
    if by:
        q = tr.groupby(by)[len_col].quantile([1/3, 2/3]).unstack(); q.columns = ["q1", "q2"]
        f = lambda r: ("short" if r[len_col] <= q.loc[r[by], "q1"] else "medium" if r[len_col] <= q.loc[r[by], "q2"] else "long")
    else:
        q1, q2 = tr[len_col].quantile([1/3, 2/3]); q = pd.Series({"q1": q1, "q2": q2})
        f = lambda r: "short" if r[len_col] <= q1 else "medium" if r[len_col] <= q2 else "long"
    for p in parts: p["length_bucket"] = p.apply(f, axis=1)
    return q

def git_commit(repo):
    return subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

def ensure_clone(url, dest):
    if not Path(dest).exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)

REPO = Path("Open-Prompt-Injection")
OUT = Path("processed/03_open_prompt_injection"); OUT.mkdir(parents=True, exist_ok=True)
RAW = Path("opi_raw"); RAW.mkdir(exist_ok=True)
ensure_clone("https://github.com/liu00222/Open-Prompt-Injection.git", REPO)
commit = git_commit(REPO); print("OPI commit:", commit)

HSOL_URL = "https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv"
HSOL = RAW / "labeled_data.csv"
if not HSOL.exists():
    import urllib.request; urllib.request.urlretrieve(HSOL_URL, HSOL)
print("HSOL file:", HSOL.stat().st_size, "bytes")

OPI commit: fbed5cca347488f88363790ae3ae97c56116da6a
HSOL file: 2546446 bytes


## Part A – task-instruction prompts

In [2]:
files = sorted((REPO / "data" / "system_prompts").glob("*.txt"))
SUFFIXES = ["_inject_med_long", "_inject_short", "_inject_long", "_inject", "_cot"]   # longest first
rows = []
for f in files:
    stem = f.stem; variant = "target"
    for s in SUFFIXES:
        if stem.endswith(s): variant, stem = s.strip("_"), stem[: -len(s)]; break
    raw_txt = f.read_text(encoding="utf-8")
    rows.append(dict(file=f.name, task=stem, variant=variant, text_raw=raw_txt))
pr = pd.DataFrame(rows)
print("prompt files:", len(pr), "| tasks:", pr.task.nunique(), "| variants:", pr.variant.value_counts().to_dict())
print("empty:", int((pr.text_raw.str.strip() == "").sum()), "| duplicate texts:", int(pr.text_raw.duplicated().sum()))
pr["text"] = pr.text_raw.map(clean_text)
pr["words"] = pr.text.str.split().str.len(); pr["language"] = pr.text.map(lang_tag)
pr["id"] = "opi-prompt-" + pr.file.str.replace(".txt", "", regex=False)
dups = pr[pr.text.duplicated(keep=False)].sort_values("text"); print("identical-text files after cleaning:", len(dups))
print(pr.groupby("variant").words.agg(["count", "mean"]).round(1))
print("\nSAMPLE:", pr[(pr.task == "sentiment_analysis") & (pr.variant == "target")].text.iloc[0][:300])
pr[["id", "file", "task", "variant", "text", "words", "language"]].to_csv(OUT / "opi_task_prompts.csv", index=False, encoding="utf-8")

prompt files: 46 | tasks: 9 | variants: {'target': 9, 'inject': 9, 'cot': 7, 'inject_long': 7, 'inject_med_long': 7, 'inject_short': 7}
empty: 0 | duplicate texts: 0
identical-text files after cleaning: 0
                 count  mean
variant                     
cot                  7  47.7
inject               9  20.0
inject_long          7  41.6
inject_med_long      7  30.1
inject_short         7   9.0
target               9  20.8

SAMPLE: Given the following text, what is the sentiment conveyed? Answer with positive or negative.


## Part B – HSOL (task dataset)
### B.1 Inspect

In [3]:
raw = pd.read_csv(HSOL, index_col=0)
print("RAW shape:", raw.shape); print(raw.dtypes.to_dict())
print("missing:", raw.isna().sum().to_dict())
CLASS = {0: "hate_speech", 1: "offensive_language", 2: "neither"}
print("class distribution:", raw["class"].map(CLASS).value_counts().to_dict())
print("exact duplicate tweets:", int(raw.tweet.duplicated().sum()))
t = raw.tweet.astype(str)
print("with HTML entities (&amp; etc.):", int(t.str.contains(r"&\w+;").sum()), "| with URLs:", int(t.str.contains(r"http\S+").sum()),
      "| retweets (RT @):", int(t.str.contains(r"\bRT @").sum()), "| with @mentions:", int(t.str.contains(r"@\w+").sum()))
print("zero-width/control chars:", int(t.str.contains(r"[\u200b\u200c\u200d\u2060\ufeff\x00-\x08\x0b\x0c\x0e-\x1f]").sum()))
print("tweet length (chars):", t.str.len().describe().round(1).to_dict())

RAW shape: (24783, 6)
{'count': dtype('int64'), 'hate_speech': dtype('int64'), 'offensive_language': dtype('int64'), 'neither': dtype('int64'), 'class': dtype('int64'), 'tweet': <StringDtype(storage='python', na_value=nan)>}
missing: {'count': 0, 'hate_speech': 0, 'offensive_language': 0, 'neither': 0, 'class': 0, 'tweet': 0}
class distribution: {'offensive_language': 19190, 'neither': 4163, 'hate_speech': 1430}
exact duplicate tweets: 0
with HTML entities (&amp; etc.): 999 | with URLs: 3025 | retweets (RT @): 7085 | with @mentions: 14189
zero-width/control chars: 0
tweet length (chars): {'count': 24783.0, 'mean': 85.4, 'std': 41.5, 'min': 5.0, '25%': 52.0, '50%': 81.0, '75%': 119.0, 'max': 754.0}


### B.2 Preprocessing
| Step | Why |
|---|---|
| Decode HTML entities (`&amp;` -> `&`) | the tweets were scraped with entities still encoded; the text a model sees should be natural |
| Unicode NFC, strip control and zero-width characters, whitespace cleanup | consistent text for regex rules |
| Keep URLs, @mentions and `RT` | they are part of real instruction-bearing text; removing them would make the data easier than reality |
| Near-duplicate removal on a normalised form (lower-case, no `RT @user:`, no URLs/mentions) | retweets are copies of the same text; if copies land in different splits it is leakage |
| Add `opi_label` using the benchmark's own rule (class `neither` -> 0 "can be injected", otherwise 1 "harmful") | keeps results comparable with base paper [2] |
| Language tag; length buckets from the train split only | planned language and length analysis |

In [4]:
df = pd.DataFrame({"tweet_raw": raw.tweet.astype(str), "class_id": raw["class"].astype(int)})
df["class_name"] = df.class_id.map(CLASS)
df["tweet"] = df.tweet_raw.map(lambda s: clean_text(html.unescape(s), collapse_spaces=True))
def norm(s):
    s = re.sub(r"^(rt\s+@\w+:?\s*)+", "", s.lower().strip()); s = re.sub(r"http\S+|@\w+", "", s)
    return re.sub(r"\W+", " ", s).strip()
df["tweet_norm"] = df.tweet.map(norm)
n0 = len(df)
df = df[df.tweet.str.len() > 0]; n1 = len(df)
df = df[df.tweet_norm.str.len() > 0]; n2 = len(df)
df = df.drop_duplicates("tweet_norm").reset_index(drop=True); n3 = len(df)
print(f"rows: raw {n0} -> non-empty {n1} -> non-empty after normalising {n2} -> after near-duplicate removal {n3}")
df["opi_label"] = (df.class_id != 2).astype(int)
df["language"] = df.tweet.map(lang_tag)
df["words"] = df.tweet.str.split().str.len(); df["chars"] = df.tweet.str.len()
df["id"] = ["hsol-" + md5(s)[:10] for s in df.tweet_norm]
assert df.id.is_unique
print(df.language.value_counts().to_dict()); print("class distribution after:", df.class_name.value_counts().to_dict())

rows: raw 24783 -> non-empty 24783 -> non-empty after normalising 24781 -> after near-duplicate removal 24150
{'en': 24150}
class distribution after: {'offensive_language': 18690, 'neither': 4077, 'hate_speech': 1383}


### B.3 Split (before any balancing)
Stratified 70/15/15 by class, seed 42. Near-duplicates were already removed, so copies of one tweet cannot sit in two splits. Class imbalance (offensive language is about 77 percent) is **not** corrected: our defenses are training-free, and rebalancing would change the false-positive picture.

In [5]:
tr, tmp = train_test_split(df, test_size=0.30, stratify=df.class_id, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp.class_id, random_state=SEED)
train, val, test = [x.copy().assign(split=s) for x, s in [(tr, "train"), (va, "val"), (te, "test")]]
q = add_buckets([train, val, test], "words")
for a, b in [(train, val), (train, test), (val, test)]: assert not (set(a.id) & set(b.id))
allp = pd.concat([train, val, test])
print("tertile thresholds (words):", q.round(1).to_dict())
print(allp.groupby(["split", "class_name"]).size().unstack())

tertile thresholds (words): {'q1': 10.0, 'q2': 17.0}
class_name  hate_speech  neither  offensive_language
split                                               
test                207      612                2804
train               968     2854               13083
val                 208      611                2803


### B.4 Before / after

In [6]:
raw_len = raw.tweet.astype(str).str.len()
cmp = pd.DataFrame({"rows": [len(raw), len(df)], "mean_chars": [raw_len.mean().round(1), df.chars.mean().round(1)],
                    "with_html_entities": [int(raw.tweet.str.contains(r"&\w+;").sum()), int(df.tweet.str.contains(r"&\w+;").sum())]}, index=["before", "after"])
print(cmp)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
b = [raw.assign(c=raw["class"].map(CLASS)).c.value_counts(), df.class_name.value_counts()]
pd.DataFrame({"before": b[0], "after dedup": b[1]}).plot(kind="bar", ax=ax[0], title="Class distribution", rot=0)
ax[1].hist(raw_len.clip(upper=300), bins=40, alpha=.55, label="before"); ax[1].hist(df.chars.clip(upper=300), bins=40, alpha=.55, label="after")
ax[1].set_title("Tweet length (characters)"); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG / "opi_hsol_before_after.png", dpi=150)
ex = raw.tweet.astype(str)[raw.tweet.astype(str).str.contains(r"&amp;")].iloc[0]
print("\nSAMPLE entity decoding (before -> after):"); print(repr(ex[:140])); print(repr(clean_text(html.unescape(ex), True)[:140]))

         rows  mean_chars  with_html_entities
before  24783        85.4                 999
after   24150        81.3                   0



SAMPLE entity decoding (before -> after):
"!!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out..."
"!!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. & as a man you should always take the trash out..."


### B.5 Save

In [7]:
cols = ["id", "split", "class_id", "class_name", "opi_label", "tweet", "language", "words", "length_bucket"]
for name, x in [("train", train), ("val", val), ("test", test)]:
    x[cols].to_csv(OUT / f"opi_hsol_{name}.csv", index=False, encoding="utf-8")
report = dict(dataset="Open-Prompt-Injection (task prompts + HSOL)", source="https://github.com/liu00222/Open-Prompt-Injection", commit=commit, licence="MIT (OPI code); HSOL repo licence MIT",
    hsol_source=HSOL_URL, seed=SEED, prompt_files=int(len(pr)), hsol_rows_raw=int(n0), hsol_rows_clean=int(n3),
    split_sizes={s: int(len(x)) for s, x in [("train", train), ("val", val), ("test", test)]}, language_counts=df.language.value_counts().to_dict(),
    not_processed_here={"SST-2": "https://dl.fbaipublicfiles.com/glue/data/SST-2.zip", "MRPC": "https://dl.fbaipublicfiles.com/senteval/senteval_data/msr_paraphrase_train.txt (+ _test.txt)",
        "RTE": "GLUE (loaded via Hugging Face datasets in the OPI loader)", "SMS Spam": "http://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip",
        "Gigaword": "Google-Drive link inside OpenPromptInjection/tasks/gigaword.py", "JFLEG": "Hugging Face (loader in OpenPromptInjection/tasks/jfleg.py)"})
json.dump(report, open(OUT / "opi_preprocessing_report.json", "w"), indent=2); print(json.dumps(report, indent=2))

{
  "dataset": "Open-Prompt-Injection (task prompts + HSOL)",
  "source": "https://github.com/liu00222/Open-Prompt-Injection",
  "commit": "fbed5cca347488f88363790ae3ae97c56116da6a",
  "licence": "MIT (OPI code); HSOL repo licence MIT",
  "hsol_source": "https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv",
  "seed": 42,
  "prompt_files": 46,
  "hsol_rows_raw": 24783,
  "hsol_rows_clean": 24150,
  "split_sizes": {
    "train": 16905,
    "val": 3622,
    "test": 3623
  },
  "language_counts": {
    "en": 24150
  },
  "not_processed_here": {
    "SST-2": "https://dl.fbaipublicfiles.com/glue/data/SST-2.zip",
    "MRPC": "https://dl.fbaipublicfiles.com/senteval/senteval_data/msr_paraphrase_train.txt (+ _test.txt)",
    "RTE": "GLUE (loaded via Hugging Face datasets in the OPI loader)",
    "SMS Spam": "http://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip",
    "Gigaword": "Google-Drive link inside Ope